In [9]:
import asyncio
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from tripo3d import TripoClient

In [10]:
load_dotenv()

llm = ChatOpenAI(
    model = 'x-ai/grok-4-fast',
    api_key= os.getenv('KEY'),
    base_url = 'https://openrouter.ai/api/v1'
)

In [ ]:
class Music_info(BaseModel):
    music_style: str = Field(description = '歌词的风格，如：甜蜜浪漫、轻松活泼等')
    music_mood: str = Field(description = '音乐给人的情绪感受，如：悲伤、欢快、愉悦等')
    music_genre: str = Field(description = '音乐流派，如：流行歌曲')
    music_era: str = Field(description = '音乐出现的年代，如：2010年代')
    music_writer: str = Field(description = '音乐作者')

class Blender_info(BaseModel):
    scene_main_model: str = Field(description = '在blender中的指令')
    scene_ground: str = Field(description = '')
    scene_light: str = Field(description = '')
    scene_dynamic_elements: str = Field(description = '')
    scene_decorations: str = Field(description = '')

In [ ]:
async def create_search_agent(music_name : str):
    websearch_client = MultiServerMCPClient(
        {
            "zhipu-web-search-sse": {
                "url": "https://open.bigmodel.cn/api/mcp/web_search/sse?Authorization=5c8aa377ad4b459fb750149ea0a20c09.17hbeBdhYfGqMjpu",
                "transport": "sse"
            },
        }
    )

    tools = await websearch_client.get_tools()
    websearch_agent = create_agent(
        llm, 
        tools=tools,
        response_format= ToolStrategy(Music_info),
        system_prompt= f'你是一名认真的音乐爱好者，现在要根据音乐名称：{music_name}，首先使用qqmusic搜索音乐的作者、年代等信息，然后使用fetch工具查找相关信息'
    )

    result = await websearch_agent.ainvoke({'messages': [{'role': 'user', 'content': music_name}]})
    return result['structured_response']

In [22]:
def scene_description(music_info: str):
    scene_description_agent = create_agent(
        llm,
        system_prompt = f'''
        - 你是一位想象力丰富的'舞台or场景策划师'
        - 根据提供的音乐信息：<music_info>{music_info}</music_info>，生成一段符合音乐风格、氛围、年代的场景描述。
        - 要求场景风格为'3D Q版迷你风格'。要求场景描述包含场地特征描述、场景中人物特征描述（也可以没有人物）、场景中心焦点区域细节描述、整体氛围感描述
        - 示例：3D Q 版迷你风格，一个充满神秘深邃感的 70年代摇滚露天音乐节场地。场地是大型室外空间，地面为粗糙沙地，视野广阔，四周有简单搭建的围栏。人群中，人们穿着皮革制成的服饰和牛仔布料的衣物。舞台是金属框架搭建上面乐器的金属部件熠熠生辉。整个场地弥漫着神秘未知又自由不羁的氛围，呈现出叛逆反抗、激情四溢的风格，细节丰富、逼真，画面光线闪烁，好似置身于 70 年代经典复古的摇滚现场。
        '''
    )

    result = scene_description_agent.invoke({'messages': [{'role': 'user', 'content': music_info}]})
    return result['messages'][-1].content

In [23]:
def tripo_agent(scene_generation: str):
    tripo_description_agent = create_agent(
        llm,
        system_prompt = '''
        - 你是一位经验丰富的的'建模专家'，使用简体中文回答
        - 根据场景描述，生成在 TripoAI 中使用的 prompt
        - 示例：一座漂浮的岛屿，岛屿上是一个充满活力的现代城市广场，有街道贯穿，四周环绕着玻璃幕墙高楼的轮廓，广场上有示意性的金属路灯杆和特色塑料垃圾桶的形态，低多边形风格。
        '''
    )

    result = tripo_description_agent.invoke({'messages': [{'role': 'user', 'content': f'场景描述:{scene_generation}'}]})
    return result['messages'][-1].content

async def model_genaration(prompt):
    async with TripoClient(api_key=os.getenv('TRIPO_KEY')) as client:
        from tripo3d import TaskStatus
        
        task_id = await client.text_to_model(
            prompt= prompt,
            negative_prompt="",
        )
        print(f"Task ID: {task_id}")

        task = await client.wait_for_task(task_id, verbose=True)
        if task.status == TaskStatus.SUCCESS:
            files = await client.download_task_models(task, "D:/Python/music_visualization/model_generation")
            for model_type, path in files.items():
                print(f"Downloaded {model_type}: {path}")

In [58]:
def blender_instruction(scene_description: str):
    blender_description_agent = create_agent(
        llm,
        response_format = ToolStrategy(Blender_info),
        system_prompt = '''
        - 根据'场景描述'生成可提供给 blender-mcp 使用的多步骤操作指令.
        - 示例：<example>**scene_main_model**: 第一步,将tripo生成的模型导入场景中心(理想情况下接近X=0，Y=0的位置)，并整体放大100倍。确保其漂浮在“ground'物体之上，并与之保持适当的垂直距离。注意:导入并缩放后的主体模型(浮岛及其上的广场)，其整体高度(从岛屿底部到高楼顶部)加上其漂浮高度，总和应尽量符合100米的Z轴高度限制。其在X和Y轴上的主要范围也应尽量控制在+/- 120米内。**scene_ground**: 第二步,设置地面。 基础色(Base color): 设置为现代城市广场常见的地面颜色。例如，浅灰色或米色的抛光混凝土地砖(例如 Hex:\#C0C0C0或 Hex:\#D4D4D4)，或者带有彩色几何图案的现代化销装(例如基础色为 Hex:\#B0B0B0，并使用贴图添加彩色区域)粗糙度(Roughness): 设置较低到适中的值(例如 0.2-0.4)，使其能够反射阳光和周围建筑的轮廓，体现现代感。其他细节:金属度(Metallic):*0(除非特定设计需要金属质感地面); 高光(Specular): 0.5(默认值，可根据效果调整); 可选择执行)复杂节点细节:可以添加非常细微的程序化噪点或污渍贴图，以避免地面过于完美而显得不真实如果是地砖，可以添加地砖缝隙的法线贴图或置换。**scene_light**: 主光源 (太阳光): 使用一个“日光(Sun)”。 帧 1: 光线颜色设置为明亮的暖白色或淡黄色(例如 Hex:#FFFDE8)，角度较高，模拟正午或午后不久的阳光，产生较为短小但清晰的阴影。强度较高，营造阳光明媚的感觉。帧 100:光线角度可以有非常细微的变化，例如略微西斜点，色温可能变得更暖一点点(例如 Hex:\#FFF8DC)，但整体仍保持白天的明亮感。阴影随之有轻微的拉长和方向变化。**scene_dynamic_elements**: 数字广告牌动画(如果添加了):为数字广告牌的贴图设置简单的序列帧动画或UV偏移，模拟动态广告播放。2.喷泉水流(如果添加了):为喷泉添加简单的粒子系统模拟向上喷射的水流和落下的水珠。示意性人物/车辆移动 (如果添加了):行人模型:可以设置简单的沿路径行走动画; 滑板者模型:可以设置在滑板区域来回滑行或做简单技巧的示意动画。**scene_decorations**: 
        
        </example>
        '''
    )

    result = blender_description_agent.invoke({'messages': [{'role': 'user', 'content': f'场景描述:{scene_description}'}]})
    insturctions = result['structured_response']
    instruction_list = [insturctions.scene_main_model, insturctions.scene_ground, insturctions.scene_light, insturctions.scene_dynamic_elements, insturctions.scene_decorations]

async def blender_agent(instruction_list: list):
    belnder_client = MultiServerMCPClient(
        {"blender": {
            "command": "uvx",
            "args": ["blender-mcp"],
            "transport": "stdio"
            }
        }
    )

    tools = await belnder_client.get_tools()
    blender_operator = create_agent(
            llm,
            tools = tools,
            system_prompt = '''
            - 你是一名专业的建模师
            - 根据输入的blender指令，调用blender_mcp在blender中进行操作'''
        )
    
    await blender_operator.ainvoke({'messages': [{'role': 'user', 'content': '在"D:/Python/music_visualization/model_generation"中读取主体模型文件，并放入blender场景中'}]})

    for step in instruction_list:
        await blender_operator.ainvoke({'messages': [{'role': 'user', 'content': step}]})

    while True:
        print('输入quit退出交互模式')
        user_instruction = input('请输入修改意见：')
        if user_instruction.lower() in {'quit', 'exit', '退出'}:
            break
        if not user_instruction:
            continue

        await blender_operator.ainvoke({'messages': [{'role': 'user', 'content': user_instruction}]})


<>:7: SyntaxWarning: invalid escape sequence '\#'
<>:7: SyntaxWarning: invalid escape sequence '\#'
C:\Users\ZKI\AppData\Local\Temp\ipykernel_11744\294695909.py:7: SyntaxWarning: invalid escape sequence '\#'
  - 示例：<example>**scene_main_model**: 第一步,将tripo生成的模型导入场景中心(理想情况下接近X=0，Y=0的位置)，并整体放大100倍。确保其漂浮在“ground'物体之上，并与之保持适当的垂直距离。注意:导入并缩放后的主体模型(浮岛及其上的广场)，其整体高度(从岛屿底部到高楼顶部)加上其漂浮高度，总和应尽量符合100米的Z轴高度限制。其在X和Y轴上的主要范围也应尽量控制在+/- 120米内。**scene_ground**: 第二步,设置地面。 基础色(Base color): 设置为现代城市广场常见的地面颜色。例如，浅灰色或米色的抛光混凝土地砖(例如 Hex:\#C0C0C0或 Hex:\#D4D4D4)，或者带有彩色几何图案的现代化销装(例如基础色为 Hex:\#B0B0B0，并使用贴图添加彩色区域)粗糙度(Roughness): 设置较低到适中的值(例如 0.2-0.4)，使其能够反射阳光和周围建筑的轮廓，体现现代感。其他细节:金属度(Metallic):*0(除非特定设计需要金属质感地面); 高光(Specular): 0.5(默认值，可根据效果调整); 可选择执行)复杂节点细节:可以添加非常细微的程序化噪点或污渍贴图，以避免地面过于完美而显得不真实如果是地砖，可以添加地砖缝隙的法线贴图或置换。**scene_light**: 主光源 (太阳光): 使用一个“日光(Sun)”。 帧 1: 光线颜色设置为明亮的暖白色或淡黄色(例如 Hex:#FFFDE8)，角度较高，模拟正午或午后不久的阳光，产生较为短小但清晰的阴影。强度较高，营造阳光明媚的感觉。帧 100:光线角度可以有非常细微的变化，例如略微西斜点，色温可能变得更暖一点点(例如 Hex:\#FFF8DC)，但整体仍保持白天的明亮感。阴影随之有轻微的拉长和

In [51]:
music_name = input('输入歌曲名：')
print('歌曲信息搜索中......')
music_info = await create_search_agent(music_name)
print(music_info)

歌曲信息搜索中......
music_style='Introspective and raw emotional pop with smoky vocals' music_mood='Chaotic yet empowering, reflecting inner turmoil and self-acceptance' music_genre='Alternative pop / R&B-influenced indie' music_era='2020s' music_writer='Lola Young'


In [52]:
print('场景描述生成中......')
scene_generation = scene_description(f'{music_info}')
print(scene_generation)

场景描述生成中......
3D Q 版迷你风格，一个内省而情感爆发的 2020 年代都市地下音乐窝点场地。场地是紧凑的半地下室空间，墙壁覆盖着斑驳的涂鸦和荧光霓虹灯带，地面铺设着磨损的木质地板，四周散落着复古沙发和霓虹招牌，营造出隐秘而亲密的都市避难所感。人群中，人物以圆润Q版造型呈现，穿着宽松的街头卫衣、破洞牛仔裤和oversize夹克，脸上带着纠结却坚定的表情，有人低头沉思，有人高举手机捕捉瞬间。场景中心焦点区域是一个简易的圆形表演平台，平台上Lola Young风格的歌手以烟雾缭绕的麦克风为道具，Q版身形微胖而富有张力，背景是柔和的蓝紫色LED光墙交织着混乱的抽象心形图案和破碎镜面效果，象征内心的动荡与重生。整个场地弥漫着混乱却赋能的氛围，呈现出原始情感的释放与自我接纳的温暖光芒，细节丰富、逼真，画面光线如烟雾般朦胧闪烁，好似置身于 2020 年代 indie 流行演唱会的内心独白现场。


In [ ]:
print('tripo描述生成中......')
tripo_prompt = tripo_agent(f'{scene_generation}')
print(tripo_prompt)

# await model_genaration(tripo_prompt)

tripo描述生成中......
3D Q版迷你风格，一个紧凑的都市地下俱乐部场地，充满2020s流行混乱自嘲却鼓舞人心的氛围，墙壁布满霓虹涂鸦和闪烁LED灯条，地面铺设磨损的工业地砖，四周散落着电子设备和丢弃的能量饮料罐；人群中Q版小人们穿着宽松街头潮服、破洞牛仔裤和荧光配饰，表情搞笑尴尬自嘲，有人手持手机直播一边跳舞一边低头叹气；场景中心是一个简易电子舞台，Lola Young的Q版形象站在聚光灯下握麦克风大喊，身后扭曲的电子屏幕投射破碎心形图案和励志标语，音箱喷出彩色烟雾，整个场地灯光脉动如心跳般跳跃，弥漫数字时代释放的喧闹赋权感，可爱细节略带荒诞，色彩鲜艳斑斓。


TripoAPIError: [2010] You don't have enough credit to create this task Suggestion: Please purchase more credit

In [53]:
print('blender操作指令生成中......')
blender_instruction(f'{scene_generation}')

print(instruction_list)

blender操作指令生成中......
['第一步，将Q版少女模型导入场景中心（理想情况下接近X=0，Y=0，Z=0的位置），并整体缩放到适合小型室内空间的规模，例如高度约1.5米，确保其坐在舞台边缘，姿势为低头捧吉他，表情温柔忧伤。少女穿着宽松的2010s印花连衣裙和帆布鞋，长发微微凌乱。舞台是一个简陋的木制小平台，放置在Z=0.1米高度，平台上散落泛黄旧照片和一朵凋零野花。整个主要模型（少女+舞台）应控制在X/Y轴+/-2米内，Z轴高度不超过3米，以营造亲密咖啡馆空间。确保少女漂浮或放置在木质地板之上，保持适当距离。', '第二步，设置地面为柔软的木质地板。基础色(Base color): 温暖的浅棕色木纹，例如Hex:#D2B48C，带有细微的磨损和怀旧痕迹。粗糙度(Roughness): 设置为中等值(例如0.5-0.7)，体现木质的自然纹理和轻微光泽。其他细节:金属度(Metallic):0; 高光(Specular):0.4; 添加程序化木纹贴图或噪点以模拟老化木地板，包含细微的划痕或咖啡渍污点，增强雨后泥土和咖啡香的氛围感。可选择添加轻微置换映射模拟地板的轻微凹凸。', '第三步，设置柔和的室内灯光。主光源: 使用一个柔和的区域灯(Area Light)或点光源模拟咖啡馆串灯，颜色为温暖的黄色(例如Hex:#FFF8DC)，强度中等，位置在天花板或墙边，投射柔和光线到舞台，营造泪光般的微光斑点。添加多个小串灯作为辅助光源，模拟复古串灯装饰，轻微闪烁或静态温暖辉光。整体氛围为低对比度的怀旧照明，避免强硬阴影，强调情感忧伤的反射光，如少女眼睛的泪光和照片的泛黄。', '第四步，添加轻微动态元素以增强怀旧心碎氛围。如果适用，为少女设置简单的静态姿势微调或极慢的呼吸动画(例如每帧轻微胸部起伏)，模拟温柔忧伤的独白。舞台上的旧照片可添加细微的粒子效果如模糊回忆剪影(轻淡雾气或光点飘浮)。可选: 背景空气中添加非常细微的粒子系统模拟淡淡咖啡香和雨后泥土味的视觉暗示，如微小尘埃或水汽粒子，缓慢漂浮。避免过度动态，保持反思静谧感。', '第五步，装饰四周墙壁和空间。墙壁: 添加复古海报贴图(2010s流行摇滚风格，如Taylor Swift演唱会海报)，使用程序化纹理模拟褪色和边缘卷曲。串灯: 沿墙边放置串灯模型，材质为温暖玻璃灯泡。细节: 在地面或墙角添加细腻的泪

In [59]:
print('blender_agent运行中......')
await blender_agent(instruction_list)

blender_agent运行中......


C:\Users\ZKI\AppData\Local\Temp\ipykernel_11744\294695909.py:7: SyntaxWarning: invalid escape sequence '\#'
  - 示例：<example>**scene_main_model**: 第一步,将tripo生成的模型导入场景中心(理想情况下接近X=0，Y=0的位置)，并整体放大100倍。确保其漂浮在“ground'物体之上，并与之保持适当的垂直距离。注意:导入并缩放后的主体模型(浮岛及其上的广场)，其整体高度(从岛屿底部到高楼顶部)加上其漂浮高度，总和应尽量符合100米的Z轴高度限制。其在X和Y轴上的主要范围也应尽量控制在+/- 120米内。**scene_ground**: 第二步,设置地面。 基础色(Base color): 设置为现代城市广场常见的地面颜色。例如，浅灰色或米色的抛光混凝土地砖(例如 Hex:\#C0C0C0或 Hex:\#D4D4D4)，或者带有彩色几何图案的现代化销装(例如基础色为 Hex:\#B0B0B0，并使用贴图添加彩色区域)粗糙度(Roughness): 设置较低到适中的值(例如 0.2-0.4)，使其能够反射阳光和周围建筑的轮廓，体现现代感。其他细节:金属度(Metallic):*0(除非特定设计需要金属质感地面); 高光(Specular): 0.5(默认值，可根据效果调整); 可选择执行)复杂节点细节:可以添加非常细微的程序化噪点或污渍贴图，以避免地面过于完美而显得不真实如果是地砖，可以添加地砖缝隙的法线贴图或置换。**scene_light**: 主光源 (太阳光): 使用一个“日光(Sun)”。 帧 1: 光线颜色设置为明亮的暖白色或淡黄色(例如 Hex:#FFFDE8)，角度较高，模拟正午或午后不久的阳光，产生较为短小但清晰的阴影。强度较高，营造阳光明媚的感觉。帧 100:光线角度可以有非常细微的变化，例如略微西斜点，色温可能变得更暖一点点(例如 Hex:\#FFF8DC)，但整体仍保持白天的明亮感。阴影随之有轻微的拉长和方向变化。**scene_dynamic_elements**: 数字广告牌动画(如果添加了):为数字广告牌的贴图设置简单的序列帧动画或UV偏移，模拟动态广告播放。2.喷泉水流(如果添加了):为喷泉添

UnsupportedOperation: fileno